# Module 2 — Hydraulic Actuation (runnable)

Cylinder force/speed, the area asymmetry phi, the valve flow law, and power.

## Setup
Run this first. It defines the machine and the formulas using only Python's standard library, so the notebook runs anywhere with no `pip install`.

In [ ]:
# PKM curriculum — runnable Python reference (standard library only, no installs needed)
from math import pi, sqrt, hypot

# machine defaults (identical to the simulation engine)
B, L_CLOSED, STROKE = 0.6, 0.4, 0.6
BORE, ROD = 0.040, 0.022
SUPPLY, RELIEF = 16e6, 21e6
PUMP_MAX_FLOW, RATED_FLOW, RATED_DP = 6e-4, 2.5e-4, 3.5e6
PAYLOAD = 12.0

def inverse_kinematics(x, y, b=B): return hypot(x+b, y), hypot(x-b, y)
def forward_kinematics(L1, L2, b=B):
    x = (L1**2 - L2**2)/(4*b); return x, sqrt(max(0.0, L1**2 - (x+b)**2))
def det_jacobian(x, y, b=B):
    L1, L2 = inverse_kinematics(x, y, b); return 2*b*y/(L1*L2)
def manipulability(x, y, b=B): return abs(det_jacobian(x, y, b))
def cap_area(D=BORE): return pi*D**2/4
def rod_area(D=BORE, d=ROD): return pi*(D**2 - d**2)/4
def asymmetry(D=BORE, d=ROD): return cap_area(D)/rod_area(D, d)
def valve_flow(u, dP, Qr=RATED_FLOW, dPr=RATED_DP): return u*Qr*sqrt(max(0.0, dP/dPr))
print("reference loaded — stdlib only")

### Lesson 1.1 — Force and speed of a cylinder
F = pA and v = Q/A.

In [ ]:
A = cap_area()
print(f'A_cap = {A*1e6:.0f} mm^2')
print(f'force @16 MPa = {SUPPLY*A/1e3:.1f} kN')
print(f'speed @15 L/min = {RATED_FLOW/A:.2f} m/s')

### Lesson 1.2 — Area asymmetry phi
phi = A_cap / A_rod, always > 1.

In [ ]:
print(f'A_cap={cap_area()*1e6:.0f} mm^2, A_rod={rod_area()*1e6:.0f} mm^2')
print(f'phi = {asymmetry():.2f}')
print(f'thicker 28 mm rod -> phi = {asymmetry(0.040, 0.028):.2f}')

### Lesson 1.3 — Four defining numbers
Extend/retract force and speed at the defaults.

In [ ]:
Ac, Ar = cap_area(), rod_area()
print(f'F_ext={SUPPLY*Ac/1e3:.1f} kN  F_ret={SUPPLY*Ar/1e3:.1f} kN')
print(f'v_ext={RATED_FLOW/Ac:.2f} m/s  v_ret={RATED_FLOW/Ar:.2f} m/s')

### Lesson 2.1 — Valve flow law
Flow is linear in command u but scales with the square root of pressure drop.

In [ ]:
real   = valve_flow(0.7, RATED_DP/2)*60000
linear = 0.7*15*0.5
print(f'real (sqrt) = {real:.1f} L/min   naive linear = {linear:.2f} L/min')

### Lesson 2.3 — Pump sizing & power
P = pQ; the pump must cover the summed worst-case demand.

In [ ]:
print(f'hydraulic power = {SUPPLY*PUMP_MAX_FLOW/1e3:.1f} kW')
demand = 2 * 0.28 * rod_area()    # two legs retracting at 0.28 m/s
print(f'2-leg demand = {demand*60000:.1f} L/min  (pump supplies {PUMP_MAX_FLOW*60000:.0f})')